In [36]:
import tensorflow as tf

# 訓練データ・テストデータの作成
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(128,128),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(128,128),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

# 0:cat,1:dog

Found 300 files belonging to 2 classes.


Found 100 files belonging to 2 classes.


In [37]:
# 訓練データの水増し

# 画像の水増しをする関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

# 水増し処理を実行
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

# 水増しデータを訓練データに追加
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

# データをシャッフルする
train_dataset = train_dataset.shuffle(32)

In [38]:
# モデルを使い学習する

input_layer = tf.keras.Input(shape=(128, 128, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理をする層(入力層と結合)

# MobileNetV2モデルの作成(画像サイズをリサイズ)
base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(128, 128, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

# 出力層の作成(犬猫判断)
output_layer = tf.keras.layers.Dense(1,activation='sigmoid')

# 各層の結合
model = tf.keras.Sequential([
    base_model,
    output_layer
])

model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

In [39]:
# 学習実行
model.fit(train_dataset, epochs=20)

# 学習結果の評価(正答率)
model.evaluate(test_dataset)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 85ms/step - accuracy: 0.7883 - loss: 0.4192
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - accuracy: 0.9294 - loss: 0.1947
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9517 - loss: 0.1446
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - accuracy: 0.9639 - loss: 0.1199
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 100ms/step - accuracy: 0.9739 - loss: 0.0982
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 9s 129ms/step - accuracy: 0.9794 - loss: 0.0847
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accuracy: 0.9850 - loss: 0.0722
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.9889 - loss: 0.0639
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 0.9911 - loss: 0.0568
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 0.9922 - loss: 0.0510
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 92ms/step - accuracy: 0.9917 - loss: 0.0471
Epoch 12/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accura

[0.03612428531050682, 0.9900000095367432]